# 相对位置偏置

和绝对位置编码不同，Swin Transformer使用了相对位置偏置（Relative Position Bias）。这是一种在计算注意力权重时引入位置信息的方法，但它不直接编码绝对位置，而是根据元素之间的相对位置关系来调整注意力分数。

In [1]:
import torch
from transformers import SwinForImageClassification, SwinConfig
from torchinfo import summary

model_name = "microsoft/swin-tiny-patch4-window7-224"
config = SwinConfig.from_pretrained(model_name)  # 先加载配置
model = SwinForImageClassification.from_pretrained(
    model_name, 
    config=config,
    ignore_mismatched_sizes=True  # 忽略分类头维度不匹配（如需仅看结构，可加）
)
model.eval()

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
              (attention): SwinAttention(
                (self): SwinSelfAttention(
                  (query): Linear(in_features=96, out_features=96, bias=True)
                  (key): Linear(in_features=96, out_features=96, bias=True)
                  (value): Linear(in_features=96, out_features=96, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
                (output): SwinSelfOutput(
        

- `relative_position_bias_table`：这是一个可学习的参数表，存储了不同相对位置偏置的值。对于一个窗口大小为$M \times M$，相对位置偏置表的大小为$(2M-1) \times (2M-1)$，因为相对位置可以从`-(M-1)`到`(M-1)`变化。

In [33]:
# 定位Swin模型代码中的Attention模块
swin_stage = model.swin.encoder.layers[0]
swin_block = swin_stage.blocks[0]
attn_module = swin_block.attention.self
relative_position_bias_table = attn_module.relative_position_bias_table.detach()
print(relative_position_bias_table.shape)  # [(2M-1) * (2M-1), num_heads]

torch.Size([169, 3])


- 通过`relative_position_bias_table`计算对应的`relative_position_bias`矩阵`B`，其形状为$M^2 \times M^2$。这里的每个元素`B[k][j]`表示窗口内位置`k`和位置`j`之间的相对位置偏置。

In [70]:
def create_relative_position_index(window_size=(7, 7)):
    # 计算窗口内每个token的相对位置索引
    coords_h = torch.arange(window_size[0])
    coords_w = torch.arange(window_size[1])
    coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))
    coords_flatten = torch.flatten(coords, 1)
    relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
    relative_coords = relative_coords.permute(1, 2, 0).contiguous()
    relative_coords[:, :, 0] += window_size[0] - 1
    relative_coords[:, :, 1] += window_size[1] - 1
    relative_coords[:, :, 0] *= 2 * window_size[1] - 1
    relative_position_index = relative_coords.sum(-1)
    return relative_position_index
def hatB2B(hatB, window_size=(7, 7), relative_position_index=None):
    B = hatB[relative_position_index.view(-1)]
    B = B.view(
        window_size[0] * window_size[1], window_size[0] * window_size[1], -1
    )
    B = B.permute(2, 0, 1).contiguous()
    return B

- 相对位置索引`relative_position_index[k]`
  - 首先，$k$对应窗口内的一个位置$(x, y)$
  - 然后，$j$对应窗口内另一个位置$(x', y')$
  - 计算公式：$\operatorname{relative\_position\_index}[k][j] = (x - x' + M - 1) * (2M - 1) + (y - y' + M - 1)$

In [91]:
x, y = 6, 6
M = 7
for j in range(49):
    xp, yp = divmod(j, M)
    print((x-xp+M-1) * (2*M-1) + (y - yp + M - 1), end=" ")

168 167 166 165 164 163 162 155 154 153 152 151 150 149 142 141 140 139 138 137 136 129 128 127 126 125 124 123 116 115 114 113 112 111 110 103 102 101 100 99 98 97 90 89 88 87 86 85 84 

In [92]:
relative_position_index = create_relative_position_index()
print(relative_position_index.shape)  # [49, 49]
hatB = relative_position_bias_table # [169, 3]
B = hatB2B(
    hatB=hatB,
    window_size=(7, 7),
    relative_position_index=relative_position_index,
)
print(B.shape)

torch.Size([49, 49])
torch.Size([3, 49, 49])
